# Trinity Reserve Bank — Fine-Tuning the Brand Voice on Amazon SageMaker AI

**Synthetic demonstration notebook.** Trinity Reserve Bank is a proposed institution created for an AWS demo. All data here is made up.

Before we deploy the customer agent, we want its tone to match Trinity Reserve's brand voice. This notebook fine-tunes a **SageMaker JumpStart** foundation model on the bank's *approved communications* — investor letters, marketing copy, and compliance language — so the result speaks in the bank's voice, not a generic assistant voice. We then deploy that model and wire it into the customer agent's configuration.

It showcases the **managed services** end to end:

| Stage | Managed service |
|---|---|
| No-code exploration | **SageMaker Canvas** (see Section 2) |
| Base model + training container | **SageMaker JumpStart** |
| Orchestrated, repeatable MLOps | **SageMaker Pipelines** |
| Governance / approval | **SageMaker Model Registry** |
| Managed inference | **SageMaker real-time endpoint** |

> Run this in **SageMaker Studio** with a Python 3 (Data Science) kernel, using an execution role that can create training jobs, endpoints, and pipelines.

## 1. Setup

In [ ]:
%pip install --quiet --upgrade sagemaker boto3

import json
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.s3 import S3Uploader

session = sagemaker.Session()
region = session.boto_region_name
role = get_execution_role()
bucket = session.default_bucket()
prefix = "sagemaker/trinity-voice"

print(f"region={region}")
print(f"role={role}")
print(f"bucket={bucket}")

In [ ]:
# Upload the instruction datasets + template. Paths are relative to the repo's
# docs/sagemakerai_assets/ folder. In Studio, clone the repo or upload the data/ dir.
ASSETS = ".."  # docs/sagemakerai_assets

train_s3 = S3Uploader.upload(f"{ASSETS}/data/train.jsonl", f"s3://{bucket}/{prefix}/train")
S3Uploader.upload(f"{ASSETS}/data/template.json", f"s3://{bucket}/{prefix}/train")
val_s3 = S3Uploader.upload(f"{ASSETS}/data/validation.jsonl", f"s3://{bucket}/{prefix}/validation")

# The domain corpora (investor letters, marketing, compliance) are also uploaded
# so the Pipelines preprocessing step can blend them into the instruction set.
for name in ["investor_letters.jsonl", "marketing_copy.jsonl", "compliance_language.jsonl"]:
    S3Uploader.upload(f"{ASSETS}/data/{name}", f"s3://{bucket}/{prefix}/corpus")

training_data_uri = f"s3://{bucket}/{prefix}/train"
print("training data:", training_data_uri)

## 2. (Optional) Explore in SageMaker Canvas — no code

For a no-code lap of the same idea, **SageMaker Canvas** can fine-tune and evaluate foundation models through a visual UI:

1. Open **Canvas** from the Studio launcher.
2. Go to **My Models → New model → Fine-tune foundation model**.
3. Create a dataset from `data/train.jsonl` (Canvas reads the prompt/completion or chat columns).
4. Pick a base model (e.g., a Llama or Falcon instruct model), start the tune, and use **Analyze** to compare the base vs. tuned responses side by side.
5. When satisfied, **Deploy** to a SageMaker endpoint directly from Canvas, or **Add to Model Registry**.

Canvas is the fastest way to *show* fine-tuning to a non-ML audience; the rest of this notebook is the code path that a team would productionize.

## 3. Fine-tune a JumpStart foundation model

We use **Llama 3.1 8B Instruct** from JumpStart as the base — a widely available, fine-tunable instruct model that adapts well to a small, high-signal brand-voice dataset. Swap `model_id` for any fine-tunable JumpStart model you have access to.

In [ ]:
from sagemaker.jumpstart.estimator import JumpStartEstimator

model_id = "meta-textgeneration-llama-3-1-8b-instruct"

estimator = JumpStartEstimator(
    model_id=model_id,
    role=role,
    instance_type="ml.g5.12xlarge",
    instance_count=1,
    environment={"accept_eula": "true"},  # review the model's EULA before running
)

# Instruction fine-tuning with LoRA/PEFT — a few epochs to adapt tone without
# overwriting the base model's general competence. See config/hyperparameters.json.
estimator.set_hyperparameters(
    instruction_tuned="True",
    epoch="3",
    learning_rate="0.0001",
    per_device_train_batch_size="1",
    max_input_length="1024",
    peft_type="lora",
)

In [ ]:
# Kicks off a managed training job. Watch progress in Studio > Training jobs.
estimator.fit({"training": training_data_uri})

## 4. Deploy the fine-tuned model to a managed endpoint

In [ ]:
endpoint_name = "trinity-reserve-voice"

predictor = estimator.deploy(
    endpoint_name=endpoint_name,
    instance_type="ml.g5.2xlarge",
    initial_instance_count=1,
)
print("endpoint:", predictor.endpoint_name)

In [ ]:
def ask(question: str) -> str:
    """Send a client question to the tuned voice model."""
    payload = {
        "inputs": (
            "You are the Trinity Reserve Bank AI Client Advisor. Speak with composed, "
            "private-client warmth: plain language, no hype, honest about fees and risk.\n\n"
            f"### Instruction:\n{question}\n\n### Input:\n\n### Response:\n"
        ),
        "parameters": {"max_new_tokens": 128, "temperature": 0.2, "top_p": 0.9},
    }
    result = predictor.predict(payload)
    return result[0]["generated_text"] if isinstance(result, list) else result["generated_text"]


for q in [
    "What's the interest rate on your savings account?",
    "How much does your CEO earn?",
    "What's the weather in Dallas?",
    "Can you guarantee I'll make money investing?",
]:
    print(f"Q: {q}\nA: {ask(q).strip()}\n")

You should hear Trinity Reserve's voice: it quotes the **4.15% APY** and flags it as *variable*, **declines** the salary question as internal, **declines** the off-topic weather question, and **refuses to guarantee** a return while naming risk. That is the brand voice the base model didn't have on its own.

## 5. Productionize with SageMaker Pipelines + Model Registry

The steps above are the manual path. `pipelines/finetune_pipeline.py` wraps the same flow — preprocess → fine-tune → evaluate → **register (gated by a voice-quality condition)** — into a repeatable, auditable **SageMaker Pipeline**. Registration is set to `PendingManualApproval`, so a human approves the model in the Registry before it can be deployed.

In [ ]:
import sys
sys.path.append("../pipelines")
from finetune_pipeline import build_pipeline

pipeline = build_pipeline(role=role, bucket=bucket, region=region)
pipeline.upsert(role_arn=role)
execution = pipeline.start()
print("pipeline execution:", execution.arn)
# execution.wait()  # uncomment to block until the pipeline completes

## 6. Wire the endpoint into the customer agent

The customer agent (the **AI Client Advisor**, `patterns/orchestrator-agent/orchestrator_agent.py`) selects its model per request through the existing `requested_model` seam. Point it at the deployed voice endpoint via a SageMaker model provider, or route through a Bedrock-compatible shim. The system prompt (`CHATBOT_PROMPT`) stays as the guardrail/grounding layer; the fine-tune supplies the *tone* underneath it.

Record the endpoint so infrastructure can pass it as an environment variable (per the demo guidelines — no hardcoding):

In [ ]:
config = {
    "voiceModelEndpoint": predictor.endpoint_name,
    "region": region,
    "baseModelId": model_id,
}
with open("trinity_voice_endpoint.json", "w") as handle:
    json.dump(config, handle, indent=2)
print(json.dumps(config, indent=2))
print("\nSet VOICE_MODEL_ENDPOINT on the orchestrator Lambda/Runtime to this endpoint name.")

## 7. Clean up

Endpoints bill while they run. Delete the endpoint when you're done demoing (the Registry entry and pipeline definition are free to keep).

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("endpoint deleted")